In [1]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import json
import re
from pathlib import Path

Предподготовка

In [2]:
ALIASES = {
    "город федерального значения севастополь": "севастополь",
    "кемеровская область - кузбасс": "кемеровская область",
    "чувашская республика - чувашия": "чувашская республика",
    "республика адыгея адыгея": "республика адыгея",
    "республика северная осетия - алания":
        "республика северная осетия - алания",
    "ханты-мансийский ао - югра":
        "ханты-мансийский автономный округ - югра",
}
SHORT_REGION_NAMES = {
    "еврейская ао": "еврейская автономная область",
    "ненецкий ао": "ненецкий автономный округ",
    "ханты-мансийский ао - югра": "ханты-мансийский автономный округ - югра",
    "чукотский ао": "чукотский автономный округ",
    "ямало-ненецкий ао": "ямало-ненецкий автономный округ",
}
def normalize_region_name(name):
    if not isinstance(name, str):
        return ""

    name = name.lower().strip()
    name = name.replace("ё", "е")
    name = name.replace("—", "-")
    name = name.replace("–", "-")
    name = re.sub(r"\s+", " ", name)

    # Сначала обрабатываем регионы, где АО означает область или округ.
    name = SHORT_REGION_NAMES.get(name, name)

    # Точка в сокращении находится перед концом строки, поэтому здесь
    # нельзя ставить \b после необязательной точки.
    name = re.sub(r"\bобл\.(?=\s|$)|\bобл\b", "область", name, flags=re.IGNORECASE)
    name = re.sub(r"\bао\.(?=\s|$)|\bао\b", "автономный округ", name, flags=re.IGNORECASE)
    name = re.sub(r"\s+", " ", name)

    # Убираем уточнение в скобках:
    # Республика Татарстан (Татарстан) -> Республика Татарстан
    name = re.sub(r"\s*\([^)]*\)", "", name)

    # Убираем начало "г."
    name = re.sub(r"^г\.\s*", "", name)

    return name.strip()


def get_region_code(region_name):
    normalized_name = normalize_region_name(region_name)
    canonical_name = ALIASES.get(normalized_name, normalized_name)
    return name_to_code.get(normalize_region_name(canonical_name))

Очистка базы клиник

In [12]:
med_df = pd.read_csv("../data/raw/med-raw.csv", sep=";")
print("Всего строк:", len(med_df))

clean_df = med_df[med_df["Страна"] == "Россия"]

clean_df["ID"] = pd.to_numeric(
    clean_df["Карточка организации"].str.extract(r"/(\d+)$")[0],
    errors="coerce"
)
clean_df = clean_df.drop_duplicates(subset=["ID"], keep="first")

clean_df.loc[clean_df["Город"] == "Севастополь", "Регион"] = "Севастополь"

clean_df["Широта"] = pd.to_numeric(
    clean_df["Широта"],
    errors="coerce"
)

clean_df["Рейтинг"] = pd.to_numeric(
    clean_df["Рейтинг"]
    .astype(str)
    .str.replace(",", ".", regex=False),
    errors="coerce"
)
clean_df = clean_df[clean_df["Рейтинг"] != 0.0]
clean_df = clean_df[(clean_df["Оценок"] > 0)]

clean_df = clean_df.dropna(subset=["Широта", "Долгота", "Регион"])

print("Строк после очистки нулей:", len(clean_df))

Всего строк: 20643
Строк после очистки нулей: 17577


C:\Users\Misha\AppData\Local\Temp\ipykernel_2996\3230173837.py:1: DtypeWarning: Columns (4,27) have mixed types. Specify dtype option on import or set low_memory=False.
  med_df = pd.read_csv("../data/raw/med-raw.csv", sep=";")
C:\Users\Misha\AppData\Local\Temp\ipykernel_2996\3230173837.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  clean_df["ID"] = pd.to_numeric(


In [13]:
# Путь к GeoJSON с границами регионов
geojson_path = (
    Path("../Russia_geojson_OSM")
    / "GeoJson's"
    / "Countries"
    / "Russia_regions.geojson"
)

regions_gdf = gpd.read_file(geojson_path)

# В GeoJSON названия регионов находятся в поле "region"
regions_gdf = regions_gdf[["region", "geometry"]].copy()
regions_gdf = regions_gdf.rename(columns={"region": "Регион_по_координатам"})
regions_gdf = regions_gdf.set_crs("EPSG:4326", allow_override=True)

# Сохраняем исходные значения на случай, если регион не определится
clean_df["Регион_исходный"] = clean_df["Регион"]

# Приводим координаты к числам
clean_df["Широта_num"] = pd.to_numeric(
    clean_df["Широта"],
    errors="coerce"
)

clean_df["Долгота_num"] = pd.to_numeric(
    clean_df["Долгота"],
    errors="coerce"
)

# Берём только строки с корректными координатами
has_coordinates = clean_df["Широта_num"].notna() & clean_df["Долгота_num"].notna()

points_gdf = gpd.GeoDataFrame(
    clean_df.loc[has_coordinates].copy(),
    geometry=gpd.points_from_xy(
        clean_df.loc[has_coordinates, "Долгота_num"],
        clean_df.loc[has_coordinates, "Широта_num"]
    ),
    crs="EPSG:4326"
)

# Определяем регион, в полигоне которого находится здание
joined = gpd.sjoin(
    points_gdf,
    regions_gdf,
    how="left",
    predicate="within"
)

# Обновляем только те строки, где регион найден в GeoJSON
matched = joined["Регион_по_координатам"].notna()

clean_df.loc[
    joined.index[matched],
    "Регион"
] = joined.loc[
    matched,
    "Регион_по_координатам"
].to_numpy()

# Удаляем технические колонки
clean_df = clean_df.drop(
    columns=["Широта_num", "Долгота_num", "Регион_по_координатам", "Регион_исходный"],
    errors="ignore"
)
print("Всего строк:", len(clean_df))
print("Регион определён по координатам:", matched.sum())
print("Регион сохранён без изменений:", len(clean_df) - matched.sum())
clean_df.to_csv("../data/clean/med-clean.csv", index=False, encoding="utf-8-sig", sep=";", decimal=",")

Всего строк: 17577
Регион определён по координатам: 17550
Регион сохранён без изменений: 27


In [15]:
med_urls = clean_df[["Карточка организации"]].copy()
med_urls.to_csv("../data/clean/med-urls.csv", index=False, encoding="utf-8-sig", sep=";", decimal=",")